# generate_config() and challenge_configurations_under_uncertainty() — test on 10 Low-Rise buildings

Step 1, one call to `generate_config()` (in `generate_config.py`, same folder), goes from the inputs to all outputs:
generate the pool → evaluate it at the baseline context → select N diverse configurations (inside the function) → re-run **only the selected ones** under the
3 economic contexts → `summary_df` for the Monte-Carlo pipeline + figures.
Step 2, `challenge_configurations_under_uncertainty()` (in `challenge_configurations.py`), confronts the selected designs with 1000 sampled futures.

REHO results are saved as they are produced: if the call is interrupted, calling it again resumes where it stopped.

`QUICK` (cell *Inputs*) chooses the size of the design pool:
- `QUICK = False` (default, used for the results): full lever set `gc.DEFAULT_KPIS_OBJECTIVE_LINK`, 42 pool designs → about 42 + 42 + 3N REHO runs (≈ 2.5 h with Gurobi on a laptop), results in `results/LowRise10_gc/`.
- `QUICK = True`: reduced lever set (16 pool designs, about 16 + 16 + 3N runs, ≈ 45 min), only to check that every stage runs; results in `results/LowRise10_gc_quick/`.

In [ ]:
import importlib
import os
import sys

import pandas as pd

SCRIPTS_DIR = os.path.abspath('../..')                       # scripts/: uncertainty_postprocess.py
UNCERTAINTY_DIR = os.path.join(SCRIPTS_DIR, 'uncertainty')   # energy_storage_distributions.py + xlsx
sys.path[:0] = [SCRIPTS_DIR, UNCERTAINTY_DIR]

import energy_storage_distributions as esd
import generate_config as gc
import challenge_configurations as cc
for m in (gc, cc):                                            # pick up edits of the .py files without a restart
    importlib.reload(m)

pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 40)

## Inputs

In [ ]:
DISTRICT = 'LowRise'                                      # district name, used in file names
QB_FILE = '../results/district_candidates_qbuildings.pickle'  # buildings read from this pickle (no QBuildings DB connection)

# 10 Low-Rise buildings (drawn once with seed 2026, then fixed so every run uses the same ones)
BUILDINGS = ['1004675', '1004681', '1004687', '1004709', '1004710', '1004715', '1004724',
             '1004729/1004731', '2035402', '3131478/295075441/295092672']

WEATHER = {'Location': 'Geneva', 'Attributes': ['T', 'I', 'W'], 'Periods': 10, 'PeriodDuration': 24}  # 10 typical days
CONTEXTS = gc.DEFAULT_CONTEXTS                    # 3 economic contexts (RT, FI): attractive / baseline / harsh
UNCERTAIN_VAR_LINK = gc.DEFAULT_UNCERTAIN_VAR_LINK  # uncertain variable -> design result it acts on (for the figures)

# QUICK = False: full design pool (all levers of gc.DEFAULT_KPIS_OBJECTIVE_LINK) -> results used in the paper.
# QUICK = True : reduced pool (2 eps, 1 SS share, 1 tariff pair, 1 value per size) -> fast check that every stage
#                runs. Each mode writes to its own OUT_DIR, so the two never overwrite each other.
QUICK = False
if QUICK:
    LEVERS = {'TOTEX': {'lever': 'mga', 'eps': [0.1, 0.5], 'objectives': ['CAPEX_min', 'CAPEX_max', 'SS_max']},
              'SS': {'lever': 'ss_target', 'fractions': [0.5]},
              'price': {'lever': 'tariffs', 'pairs': [(0.40, 0.07)]},
              'installed_capacity': {'lever': 'sizes', 'rSOC_district': [6], 'Battery_district': [100],
                                     'roof_share': [0.5], 'tech_sets': ['no_storage', 'battery_only', 'rsoc_only']},
              'GU': None}
    OUT_DIR = f'results/{DISTRICT}10_gc_quick'
else:
    # 5 MGA eps x 3 objectives, 3 SS shares, 10 (RT, FI) pairs, 3 storage sets + 4 rSOC + 3 battery sizes + 2 roof shares
    LEVERS = gc.DEFAULT_KPIS_OBJECTIVE_LINK
    OUT_DIR = f'results/{DISTRICT}10_gc'

# Number of selected designs: 'auto' = smallest N >= 10 meeting the diversity rule, at most 20 (if the rule asks for
# more, the 20 that cover the 4 KPIs best are kept); an int = upper limit instead of 20; None = keep every design.
N_MAX = 'auto'

ARGS = dict(district_id=DISTRICT, uncertain_var_link=UNCERTAIN_VAR_LINK, KPIs_objective_link=LEVERS, n_max=N_MAX,
            contexts=CONTEXTS, out_dir=OUT_DIR, qb_file=QB_FILE, buildings=BUILDINGS, weather=WEATHER,
            postprocess_dir=SCRIPTS_DIR)   # postprocess_dir: folder of the supervisor's uncertainty_postprocess.py
print(OUT_DIR)

Uncertainty distributions (supervisor's sheet). Used by step 2; `TECH_FROM_XLSX = True` also takes the design-stage
technology costs from their medians (`gc.DEFAULT_TECH` already holds the same values, so the default is `False`;
switching it needs a new `OUT_DIR`).

In [ ]:
# Distribution names used in the xlsx -> names understood by energy_storage_distributions.py
ALIASES = {'normal': 'normal', 'log-normal': 'lognormal', 'lognormal': 'lognormal', 'log normal': 'lognormal',
           'uniform': 'uniform', 'triangular': 'triangular'}
# DISTS: one distribution per uncertain variable; the xlsx low/high values are read as the 2.5 % / 97.5 % percentiles
DISTS = esd.build_distribution_dict(esd.load_parameters(os.path.join(UNCERTAINTY_DIR, 'energy_storage_parameters.xlsx'),
                                                        ALIASES), percentiles=(0.025, 0.975))

# False: design-stage technology costs = gc.DEFAULT_TECH (already equal to the xlsx medians)
# True : take them from the medians of DISTS (needs a new OUT_DIR, the designs would change)
TECH_FROM_XLSX = False
if TECH_FROM_XLSX:
    ARGS['uncertainty_distributions'] = DISTS
print(gc.tech_from_distributions(DISTS) if TECH_FROM_XLSX else gc.DEFAULT_TECH)

## Run

In [ ]:
# Step 1: generate the pool, select N designs, re-run them under the 3 contexts (resumes if interrupted)
out = gc.generate_config(**ARGS)

## Design codes

Every generated design is named `G<generator>-<nn>`; `nn` follows the order of the lever specification in `LEVERS`.

| Code | Generator | Designs |
|---|---|---|
| G0 | Anchors | G0-01 TOTEX optimum, G0-02 SS maximum |
| G1 | TOTEX optimum under other tariffs (RT / FI) | one per tariff pair |
| G2 | MGA: TOTEX ≤ TOTEX_opt + ε · |TOTEX_opt|, then min CAPEX / max CAPEX / max SS | for each ε: CAPEX_min, CAPEX_max, SS_max |
| G3 | Grid import fixed at a share of the way from the TOTEX optimum to the SS maximum | one per share |
| G4 | TOTEX optimum with a restricted storage set | no storage, battery only, rSOC only |
| G5 | TOTEX optimum with a forced minimum size or a reduced PV roof share | rSOC ≥ x, battery ≥ x, roof share |

The `description` column of the tables says exactly which run each code is.

**Clusters** = storage set actually installed in the design (size > 0): no storage, battery, rSOC, battery+rSOC (one colour each). The **name** lists the installed technologies among PV, HP, battery and rSOC, e.g. PV+HP+battery, HP (no storage) when no PV is installed, grid only when nothing is installed. A design generated as "rSOC only" can therefore end up as *PV+HP (no storage)* if the optimizer installs no rSOC.

## Selected configurations by cluster

Only the N selected designs (the rejected pool designs are not kept in the outputs and are not run under the 3 contexts).
Left: cluster (technology class); then the design code; then what the design is, KPIs and installed capacities (evaluated at the baseline context).

In [ ]:
out['cluster_table'].round(2)

## Outputs

1. The 3 × N optimization results (pickles in `OUT_DIR`, one per design × context)

In [ ]:
out['files']

2. Dataframe for the Monte-Carlo simulation (design first, then context) and which column each uncertain variable acts on

In [ ]:
sd = out['summary_df']
print(sd.shape)
print(out['uncertain_var_columns'])
sd[['techno', 'retail_elec', 'feed_in_elec', 'Elec_import', 'Elec_export', 'Battery_daily_size', 'rSOC_size',
    'CH4_storage_size', 'TOTEX_kpi', 'CAPEX_kpi', 'SS_kpi', 'GU_kpi']].round(2)

3. Selection report: coverage of every KPI / capacity, as a share of the pool range

In [ ]:
sel = out['selection']
print('selected', len(sel['selected']), 'of', sel['n_pool'], 'generated | auto N =', sel['n_auto'], '| rule met:', sel['rule_met'])
print('near-duplicates dropped:', sel['dropped_duplicates'])
sel['report'].round(2)

4. Figures (also saved as HTML in `OUT_DIR/figures`); colour = technology class.

- `kpis_vs_capex`: TOTEX, SS and GU (y) against CAPEX (x) for every selected design × context (symbol = context).
- `indicator_ranges`: range of SS, GU, TOTEX, CAPEX and of every installed capacity per technology class (baseline context; capacities are fixed by the design).
- `totex_bars`: all selected designs in one figure, REHO cost template (CAPEX left / OPEX right / thin TOTEX), grouped by technology class (shaded band), sorted by CAPEX inside each class, baseline context; comfort penalty shown explicitly.
- `uncertain_var_link`: one bar subplot per entry of `UNCERTAIN_VAR_LINK` (uncertain variable → what it acts on), designs grouped by technology class; operation results (import, export) one bar per context (pattern), installed capacities one bar (baseline context).

In [ ]:
for name, fig in out['figures'].items():
    fig.show()

## Step 2: challenge_configurations_under_uncertainty()

Inputs: the uncertainty distributions (supervisor's xlsx) and the configuration dataframe `out['summary_df']` (N designs × 3 operations).
Paired Monte Carlo (same draws for every design, supervisor's `uncertainty_postprocess.py`), all costs in CHF/m²/yr. Per draw:

- **cost** of a design = TOTEX of the operation with the best score (TOTEX, SS, GU) in that draw (`operation='best'`) or the mean of its 3 operations (`operation='blend'`)
- **adaptability** = TOTEX with the baseline operation − TOTEX with the cheapest operation (what re-optimising the operation recovers)
- **lock-in** = TOTEX with the cheapest operation − TOTEX of the cheapest design (with its cheapest operation) (what stays lost even with the best operation)
- **regret** = cost − TOTEX of the cheapest design (with its cheapest operation)
- **score** (TOTEX, SS, GU): in every draw each KPI is scaled over the designs to 0 (best) .. 1 (worst), score = weighted sum (`kpi_weights`, default 1/3 each); SS and GU are those of the operation used in that draw
- **ranking**: rank of the score in each draw → probability of each rank, P(best), expected rank
- **blend (main)**: weight of a design = share of the sampled contexts in which it has the best score (P(best)); `ch['weights']`, TOTEX, SS and GU of the blend per draw in `ch['per_draw']['blend']`
- **blend (supplement)**: risk-averse weights minimising (1 − λ)·mean + λ·CVaR(`alpha`) of the score regret; `ch['frontier']` (every λ), `ch['weights_cvar']` (λ = 1); numbers only, no figure

Figures: `capex_totex` (median and 95 % interval of each axis), `capex_totex_cloud` (all draws + region holding 95 % of them), `ranking` (blend weights | TOTEX, SS, GU of every design vs the blend; rank probabilities in `ch['rank_probability']`), `regret` (adaptability vs lock-in).

In [ ]:
# n_runs           : number of Monte-Carlo draws (same draws for every design)
# operation        : 'best' = in each draw a design uses the operation (context run) with the best score;
#                    'blend' = mean of its 3 operations
# initial_operation: operation used as the starting point for adaptability ('baseline' context)
# kpi_weights      : weights of TOTEX, SS and GU in the score (each KPI scaled 0 best .. 1 worst per draw)
# alpha            : CVaR level of the supplementary risk-averse blend (numbers only)
CH_ARGS = dict(n_runs=1000, operation='best', initial_operation='baseline',
               kpi_weights={'TOTEX': 1/3, 'SS': 1/3, 'GU': 1/3}, alpha=0.9, out_dir=OUT_DIR)

# Step 2: confront the selected designs (out['summary_df']) with the sampled futures
ch = cc.challenge_configurations_under_uncertainty(DISTS, out['summary_df'], **CH_ARGS)
print(ch['low_regret'])
ch['summary'].round(2)

In [ ]:
for name, fig in ch['figures'].items():
    fig.show()